# Getting Started with QuantX Engine

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/01_getting_started.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/01_getting_started.ipynb)


This notebook introduces the foundational capabilities of QuantX Engine:
1. **Lanczos eigensolver** — finding ground-state energies of quantum Hamiltonians
2. **Trotter time evolution** — simulating quantum dynamics
3. **Noise channels** — modeling decoherence

No C++ native library is required — all examples run in pure Python.


In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

In [ ]:
import numpy as np
import quantx_engine
print('quantx_engine version:', quantx_engine.__version__)

## 1. Lanczos Eigensolver: 2-site Heisenberg Chain

The 2-site Heisenberg Hamiltonian $H = \frac{J}{2}(X\otimes X + Y\otimes Y + Z\otimes Z)$
has exact ground-state energy $E_0 = -3J/2$ (singlet state).

In [ ]:
from quantx_engine.lanczos import LanczosSolver

J = 1.0
# H = (J/2)*(X⊗X + Y⊗Y + Z⊗Z) in the {|00>,|01>,|10>,|11>} basis
H2 = np.array([
    [ J/2,  0,    0,   0  ],
    [ 0,  -J/2,   J,   0  ],
    [ 0,    J,  -J/2,  0  ],
    [ 0,    0,    0,  J/2 ],
])
exact_e0 = -3 * J / 2   # singlet ground state

# LanczosSolver(k, max_iter, tol) — k = number of eigenvalues requested
solver = LanczosSolver(k=2, max_iter=50, tol=1e-12)
result = solver.solve(H2)

print(f'Ground-state energy E0 = {result.eigenvalues[0]:.6f}  (exact: {exact_e0:.6f})')
print(f'First excited   energy  = {result.eigenvalues[1]:.6f}  (exact: {J/2:.6f})')
assert abs(result.eigenvalues[0] - exact_e0) < 1e-6, 'Lanczos ground state incorrect!'
print('Lanczos assertion passed.')

## 2. Trotter Time Evolution

Evolve a 4-qubit Ising chain with the `ising_trotter` helper.  Norm must be conserved.

In [ ]:
from quantx_engine.trotter import ising_trotter

# Create TFIM simulator: n=4 qubits, J=1, h=0.5
ts = ising_trotter(n=4, J=1.0, h=0.5)

# run(t, n_steps, order) — t is total simulation time
result = ts.run(t=1.0, n_steps=20, order=2)

# Norm must be conserved
final_norm = np.linalg.norm(result.state)
print(f'Steps: {result.n_steps}, Order: {result.order}')
print(f'Final state norm: {final_norm:.8f}  (should be 1.0)')
assert abs(final_norm - 1.0) < 1e-6, 'Norm not conserved!'
print('Norm conservation check passed.')

## 3. Noise Channels

QuantX provides common quantum noise channels.

In [ ]:
from quantx_engine.noise import DepolarizingNoise, AmplitudeDampingNoise

# Depolarizing with p=0.1
dep = DepolarizingNoise(prob=0.1)
print(f'Depolarizing noise: prob={dep.prob}')

# Amplitude damping (T1 decay)
adc = AmplitudeDampingNoise(gamma=0.2)
print(f'Amplitude damping:  gamma={adc.gamma}')

# Verify noise parameters are in valid range
assert 0 <= dep.prob <= 1
assert 0 <= adc.gamma <= 1
print('Noise channel parameter checks passed.')

## Summary

| Module | What we demonstrated |
|--------|---------------------|
| `quantx_engine.lanczos` | Ground-state energy of 2-site Heisenberg, verified against $E_0=-3J/2$ |
| `quantx_engine.trotter` | Norm-conserving second-order Trotter evolution of 4-qubit Ising |
| `quantx_engine.noise` | Depolarizing and amplitude-damping channel construction |

Next: **[02 — VQE Chemistry](02_vqe_chemistry.ipynb)**